In [1]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# Standard Unsloth Installation
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft curate accelerate bitsandbytes unsloth_zoo
!pip install wandb 

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-h95o9bvq/unsloth_b5130c90c8a04f6cac2f67370b8a37f4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-h95o9bvq/unsloth_b5130c90c8a04f6cac2f67370b8a37f4
  Resolved https://github.com/unslothai/unsloth.git to commit 67d3519cab9fde8603d89bd6303864e6ca852cb8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.3.5-py3-none-any.whl size=29235321 sha256=af7f7f216ae1c762ab64b9dabbbcccb8ca8ac0252804459d499ae5c1c11fce13
  Stored in directory: /tmp/pip-ephem-wheel-cache-bkyt8ph4/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 
dtype = None 
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # <-- THE FIX
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

print("Llama-3.2 (3B) Loaded! Loss will definitely show numbers now.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.5: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Llama-3.2 (3B) Loaded! Loss will definitely show numbers now.


In [3]:
import os
import json
from datasets import Dataset

# 1. Find Data
raw_file = 'agrikoli_training_dataset.json'
raw_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if raw_file in files:
        raw_path = os.path.join(root, raw_file)
        break

with open(raw_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Clean Data
clean_data = []
for item in raw_data:
    if item.get("instruction") and item.get("output"):
        clean_data.append(item)

# 3. Format Task
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request in the Agri/Koli Marathi dialect.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = Dataset.from_list(clean_data)
dataset = dataset.map(formatting_prompts_func, batched = True, num_proc = 2)
print(f"Dataset ready: {len(dataset)} samples.")

Map (num_proc=2):   0%|          | 0/1020 [00:00<?, ? examples/s]

Dataset ready: 1020 samples.


In [4]:
from trl import SFTTrainer, SFTConfig
import wandb

wandb.login()

training_args = SFTConfig(
    output_dir = "outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    max_steps = 120, 
    learning_rate = 2e-4, # We can use standard speed now!
    fp16 = True,
    bf16 = False,
    logging_steps = 1,
    optim = "adamw_8bit", 
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    max_grad_norm = 1.0,
    dataset_text_field = "text",
    max_seq_length = 2048,
    report_to = "wandb",
    run_name = "agrikoli-llama-success",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

print("Starting training. Watch the loss—it's going to work!")
trainer.train()
wandb.finish()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: karanshelar8775 (karanshelar8775-student-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training. Watch the loss—it's going to work!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,020 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,4.111663
2,3.939033
3,4.068361
4,3.958758
5,3.887053
6,3.805964
7,3.484258
8,3.391594
9,2.941844
10,2.702533


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇██
train/global_step,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
train/grad_norm,▇▇▇▇▆█▄▃▄▅▅▆▆▇▆▇▄▄▃▄▃▃▄▂▂▃▁▂▂▂▅▁▁▁▁▁▁▁▁▂
train/learning_rate,▂▃▄▆▆▇██▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▂▂▂▂▂▂▂▁▁▁
train/loss,█▇▇▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁
total_flos,1428535926915072.0
train/epoch,0.94118
train/global_step,120
train/grad_norm,0.51243
train/learning_rate,0.0
train/loss,0.11612


In [5]:
# ENABLE FAST INFERENCE MODE
FastLanguageModel.for_inference(model)

# --- 🧪 TEST PROMPT ---
# Change this text to whatever you want to ask!
my_test_instruction = "How is the fishing today and how is your family? / आज मासेमारी कशी आहे आणि घरचे कसे हायेत?"

inputs = tokenizer(
[
    alpaca_prompt.format(
        my_test_instruction, # Instruction
        "", # No extra input needed
        "", # Empty response for the model to fill
    )
], return_tensors = "pt").to("cuda")

# GENERATE
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request in the Agri/Koli Marathi dialect.

### Instruction:
How is the fishing today and how is your family? / आज मासेमारी कशी आहे आणि घरचे कसे हायेत?

### Input:


### Response:


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

आज मासा ला जायचं वाढ गावलेत आज, पोटांत मासले गावलेत आज. घराचं चांगले हायेत आज.<|eot_id|>


In [6]:
model.save_pretrained("agrikoli_llama3_model") # Local saving
tokenizer.save_pretrained("agrikoli_llama3_model")
print("Model saved locally to 'agrikoli_llama3_model' folder!")

Model saved locally to 'agrikoli_llama3_model' folder!


In [7]:
import shutil
from IPython.display import FileLink

# 1. First, save the model folder (if you haven't already)
model.save_pretrained("agrikoli_llama3_model")
tokenizer.save_pretrained("agrikoli_llama3_model")

# 2. ZIP the folder into a single file
shutil.make_archive("agrikoli_llama3_model_pc", 'zip', "agrikoli_llama3_model")

# 3. Create a direct download link!
print("--- MODEL PACKAGED ---")
FileLink(r'agrikoli_llama3_model_pc.zip')

--- MODEL PACKAGED ---


/kaggle/working/agrikoli_llama3_model_pc.zip